# Props Data Organizer

## Definitions of Formulas Used in This Notebook

### Estimated Value (EV)
EV is the average amount you can expect to win or lose per bet if you placed the same bet many times. It helps identify profitable betting opportunities by comparing the expected return to the risk involved.

**Formula:**
$$
\text{EV} = (\text{Probability of Winning} \times \text{Profit if Win}) - (\text{Probability of Losing} \times \text{Loss if Lose})
$$

### Kelly Criterion
The Kelly Criterion is a formula used to determine the optimal size of a series of bets. It aims to maximize the logarithm of wealth, balancing the trade-off between risk and reward. The formula considers both the probability of winning and the odds offered, guiding you on how much of your bankroll to wager on each bet.

**Formula:**
$$
\text{Kelly Fraction} = \frac{(\text{Probability of Winning} \times (\text{Odds} + 1)) - 1}{\text{Odds}}
$$

### Variance
Variance in sports betting represents the spread or dispersion of actual outcomes around the expected value. It's a crucial metric for understanding the risk and volatility associated with betting predictions. Higher variance indicates more volatile and unpredictable outcomes, while lower variance suggests more consistent results.

**Formula:**
$$
\text{Variance} = \frac{\sum_{i=1}^{n} (x_i - \mu)^2}{n}
$$

Where:
- $x_i$ represents each individual outcome
- $\mu$ is the mean or expected value
- $n$ is the total number of observations

In the context of prop betting:
- High variance props (e.g., 3-pointers made) tend to be more risky but potentially more profitable
- Low variance props (e.g., minutes played) typically offer more consistent but lower returns


In [1]:
import pandas as pd 
import numpy as np
import time
import requests
import os
import sys
from datetime import datetime
import joblib

feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
# feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)
    
from PROPS_EV.calculateEVS import *
from MODELS.model import *

today = datetime.now()
formatted_date = today.strftime("%m_%d_%y")
pd.set_option('display.max_columns', None)

c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\venv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Grabs players odds for the day (US all boookmakers, DFS is prizepicks and underdogs)

In [2]:
# from NBAPropFinder.NBAPropFinder import NBAPropFinder

# nba_props = NBAPropFinder(region='us_dfs')
# prizePicks = nba_props.dataframe
# prizePicks.head(10)

### Single Bets from bookmakers that dont include prizePicks or UnderDogs

In [13]:
features = [
    # Player context
    'PLAYER_ID', 'TEAM_ID', 'OPP_TEAM_ID', 
    'STARTING', 'HOME_GAME', 
    'PLAYER_DAYS_REST', 'IS_BACK_TO_BACK', 
    
    # Player season averages
    'PTS_AVG_TO_DATE', 'MIN_AVG_TO_DATE', 'FGA_AVG_TO_DATE', 'FTA_AVG_TO_DATE', 'FG3A_AVG_TO_DATE', 
    'FG_PCT_AVG_TO_DATE', 'FG3_PCT_AVG_TO_DATE', 'FT_PCT_AVG_TO_DATE', 'USG_PCT_AVG_TO_DATE', 'TS_PCT_AVG_TO_DATE', 
    'EFG_PCT_AVG_TO_DATE', 'POSS_AVG_TO_DATE', 'TCHS_AVG_TO_DATE', 'AST_AVG_TO_DATE', 'REB_AVG_TO_DATE', 'TOV_AVG_TO_DATE',
    
    # LAG
    'PTS_LAG_1', 'PTS_LAG_2', 'MIN_LAG_1', 'MIN_LAG_2', 'FGA_LAG_1', 'FGA_LAG_2', 'FTA_LAG_1', 'FTA_LAG_2', 'FG3A_LAG_1', 
    'FG3A_LAG_2', 'FG_PCT_LAG_1', 'FG_PCT_LAG_2', 'FG3_PCT_LAG_1', 'FG3_PCT_LAG_2', 'FT_PCT_LAG_1', 'FT_PCT_LAG_2', 
    'USG_PCT_LAG_1', 'USG_PCT_LAG_2', 'TS_PCT_LAG_1', 'TS_PCT_LAG_2', 'EFG_PCT_LAG_1', 'EFG_PCT_LAG_2', 'POSS_LAG_1', 'POSS_LAG_2', 
    'TCHS_LAG_1', 'TCHS_LAG_2', 'AST_LAG_1', 'AST_LAG_2', 'REB_LAG_1', 'REB_LAG_2', 'TOV_LAG_1', 'TOV_LAG_2',
    
    # 3 game rolling averages
    'PTS_ROLLING_AVG_3', 'MIN_ROLLING_AVG_3', 'FGA_ROLLING_AVG_3', 'FTA_ROLLING_AVG_3', 'FG3A_ROLLING_AVG_3', 'FG_PCT_ROLLING_AVG_3', 
    'FG3_PCT_ROLLING_AVG_3', 'FT_PCT_ROLLING_AVG_3', 'USG_PCT_ROLLING_AVG_3', 'TS_PCT_ROLLING_AVG_3', 'EFG_PCT_ROLLING_AVG_3', 
    'POSS_ROLLING_AVG_3', 'TCHS_ROLLING_AVG_3', 'AST_ROLLING_AVG_3', 'REB_ROLLING_AVG_3', 'TOV_ROLLING_AVG_3',

    # Short-term form (5-game rolling averages)
    'PTS_ROLLING_AVG_5', 'MIN_ROLLING_AVG_5', 'FGA_ROLLING_AVG_5', 'FTA_ROLLING_AVG_5', 'FG3A_ROLLING_AVG_5', 'FG_PCT_ROLLING_AVG_5', 
    'FG3_PCT_ROLLING_AVG_5', 'FT_PCT_ROLLING_AVG_5', 'USG_PCT_ROLLING_AVG_5', 'TS_PCT_ROLLING_AVG_5', 'EFG_PCT_ROLLING_AVG_5', 
    'POSS_ROLLING_AVG_5', 'TCHS_ROLLING_AVG_5', 'AST_ROLLING_AVG_5', 'REB_ROLLING_AVG_5', 'TOV_ROLLING_AVG_5',
    
    # 7 game rolling averages
    'PTS_ROLLING_AVG_7', 'MIN_ROLLING_AVG_7', 'FGA_ROLLING_AVG_7', 'FTA_ROLLING_AVG_7', 'FG3A_ROLLING_AVG_7', 'FG_PCT_ROLLING_AVG_7', 
    'FG3_PCT_ROLLING_AVG_7', 'FT_PCT_ROLLING_AVG_7', 'USG_PCT_ROLLING_AVG_7', 'TS_PCT_ROLLING_AVG_7', 'EFG_PCT_ROLLING_AVG_7', 
    'POSS_ROLLING_AVG_7', 'TCHS_ROLLING_AVG_7', 'AST_ROLLING_AVG_7', 'REB_ROLLING_AVG_7', 'TOV_ROLLING_AVG_7',

    # Medium-term form (15-game rolling averages)
    'PTS_ROLLING_AVG_15', 'MIN_ROLLING_AVG_15', 'FGA_ROLLING_AVG_15', 'FTA_ROLLING_AVG_15', 'FG3A_ROLLING_AVG_15', 'FG_PCT_ROLLING_AVG_15', 
    'FG3_PCT_ROLLING_AVG_15', 'FT_PCT_ROLLING_AVG_15', 'USG_PCT_ROLLING_AVG_15', 'TS_PCT_ROLLING_AVG_15', 'EFG_PCT_ROLLING_AVG_15', 
    'POSS_ROLLING_AVG_15', 'TCHS_ROLLING_AVG_15', 'AST_ROLLING_AVG_15', 'REB_ROLLING_AVG_15', 'TOV_ROLLING_AVG_15',
    
    # Long-term form (40-game rolling averages)
    'PTS_ROLLING_AVG_40', 'MIN_ROLLING_AVG_40', 'FGA_ROLLING_AVG_40', 'FTA_ROLLING_AVG_40', 'FG3A_ROLLING_AVG_40', 'FG_PCT_ROLLING_AVG_40', 
    'FG3_PCT_ROLLING_AVG_40', 'FT_PCT_ROLLING_AVG_40', 'USG_PCT_ROLLING_AVG_40', 'TS_PCT_ROLLING_AVG_40', 'EFG_PCT_ROLLING_AVG_40', 
    'POSS_ROLLING_AVG_40', 'TCHS_ROLLING_AVG_40', 'AST_ROLLING_AVG_40', 'REB_ROLLING_AVG_40', 'TOV_ROLLING_AVG_40',

    # Opponent
    'OPP_DEF_RATING_AVG_TO_DATE', 'OPP_PACE_AVG_TO_DATE', 'OPP_PTS_AVG_TO_DATE', 'OPP_FGA_AVG_TO_DATE', 
    'OPP_REB_AVG_TO_DATE', 'OPP_AST_AVG_TO_DATE', 'OPP_TOV_AVG_TO_DATE', 'OPP_BLK_AVG_TO_DATE', 'OPP_STL_AVG_TO_DATE',
    
    #starters 
    'TEAM_OFF_RATING_AVG_TO_DATE','TEAM_DEF_RATING_AVG_TO_DATE','TEAM_PACE_AVG_TO_DATE', 'TEAM_FGA_AVG_TO_DATE',
    'TEAM_PTS_AVG_TO_DATE', 'TEAM_REB_AVG_TO_DATE', 'TEAM_AST_AVG_TO_DATE', 'TEAM_TOV_AVG_TO_DATE',
    
    # Team odds
    'team_spread', 'total', 'team_is_favored','TEAM_IMPLIED_PTS_FAV','TEAM_IMPLIED_PTS_UND','BLOWOUT_RISK'
]

# model = joblib.load('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/Models/PTS_cat_model.pkl')
model = joblib.load(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS\Models\PTS_cat_model.pkl")
data = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values('GAME_DATE', ascending=False)
bookmakers = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
prizePicks = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/odds25.csv')

date = '2024-10-27'
espnDate = '20241027'
odds = bookmakers[
    (bookmakers['CATEGORY'] == 'points') &
    (bookmakers['GAME_DATE'] == date) &
    (bookmakers['ODDS'] < 250) &
    (bookmakers['ODDS'] > -250)
]

oddsPP = prizePicks[
    (prizePicks['CATEGORY'] == 'player_points') &
    (prizePicks['GAME_DATE'] == date) &
    (prizePicks['BOOKMAKER'] == 'underdog')
]
oddsPP.rename(columns={'OVER/UNDER': 'SIDE'}, inplace=True)
filterData = data[data['GAME_DATE'] <= date].sort_values('GAME_DATE', ascending=True)
games = get_espn_games(date_str=espnDate)
oddsPP.head()

C:\Users\alexg\AppData\Local\Temp\ipykernel_7336\677740084.py:59: DtypeWarning: Columns (10,11,13) have mixed types. Specify dtype option on import or set low_memory=False.
  prizePicks = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/odds25.csv')
C:\Users\alexg\AppData\Local\Temp\ipykernel_7336\677740084.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  oddsPP.rename(columns={'OVER/UNDER': 'SIDE'}, inplace=True)


,Unnamed: 0.1,Unnamed: 0,NAME,CATEGORY,BOOKMAKER,SIDE,LINE,PRICE,HOME_TEAM,AWAY_TEAM,game_id,commence_time,GAME_DATE,period_id,fair_line,fair_odds,OVER/ODDS
4109,4109,4109,Toumani Camara,player_points,underdog,Under,7.5,-137,Portland Trail Blazers,New Orleans Pelicans,f3886316e64ad5e79bc457ce7d19a9bb,2024-10-27T22:10:00Z,2024-10-27,NaN,NaN,NaN,under
4116,4116,4116,Toumani Camara,player_points,underdog,Over,7.5,-137,Portland Trail Blazers,New Orleans Pelicans,f3886316e64ad5e79bc457ce7d19a9bb,2024-10-27T22:10:00Z,2024-10-27,NaN,NaN,NaN,over
4128,4128,4128,Javonte Green,player_points,underdog,Under,6.0,-137,Portland Trail Blazers,New Orleans Pelicans,f3886316e64ad5e79bc457ce7d19a9bb,2024-10-27T22:10:00Z,2024-10-27,NaN,NaN,NaN,under
4131,4131,4131,Javonte Green,player_points,underdog,Over,6.0,-137,Portland Trail Blazers,New Orleans Pelicans,f3886316e64ad5e79bc457ce7d19a9bb,2024-10-27T22:10:00Z,2024-10-27,NaN,NaN,NaN,over
4132,4132,4132,Zion Williamson,player_points,underdog,Under,23.5,-137,Portland Trail Blazers,New Orleans Pelicans,f3886316e64ad5e79bc457ce7d19a9bb,2024-10-27T22:10:00Z,2024-10-27,NaN,NaN,NaN,under


## Best EVs for Single Bets from draftkings, fanduel, prizepicks, and underdog

In [14]:
final_results = single_bet(filterData, odds, model, games, features, espnDate, stake=100, simulations=10000).sort_values(by='EV%', ascending=False).reset_index(drop=True)

print("\nTop 10 highest EV bets across point props:")
final_results.head(10)

Processing single bets...

DEBUG - Aaron Nesmith:
  Odds: -121 (under)
  Line: 9.5
  Prediction: 7.85
  Std Dev: 3.0
  Prob Over: 0.286
  Decimal Odds: 1.83
  Breakeven: 0.548
  Simulated Mean: 7.88
  Model Prediction: 7.85

DEBUG - Tyrese Haliburton:
  Odds: -107 (over)
  Line: 17.5
  Prediction: 14.07
  Std Dev: 9.5
  Prob Over: 0.390
  Decimal Odds: 1.93
  Breakeven: 0.517
  Simulated Mean: 15.51
  Model Prediction: 14.07

DEBUG - Myles Turner:
  Odds: -103 (over)
  Line: 16.5
  Prediction: 16.66
  Std Dev: 3.7859388972001824
  Prob Over: 0.520
  Decimal Odds: 1.97
  Breakeven: 0.507
  Simulated Mean: 16.65
  Model Prediction: 16.66

DEBUG - Andrew Nembhard:
  Odds: -108 (over)
  Line: 9.5
  Prediction: 7.64
  Std Dev: 4.163331998932266
  Prob Over: 0.333
  Decimal Odds: 1.93
  Breakeven: 0.519
  Simulated Mean: 7.93
  Model Prediction: 7.64

DEBUG - Caleb Martin:
  Odds: -103 (under)
  Line: 11.5
  Prediction: 11.33
  Std Dev: 3.214550253664318
  Prob Over: 0.487
  Decimal Odds: 1.

,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
0,James Harden,espnbet,points,21.5,-120,over,27.31,0.951,0.049,0.545,74.35,0.89,0.45,0.22,"(20.5, 34.1)"
1,Pascal Siakam,betrivers,points,20.5,-125,under,16.99,0.162,0.838,0.556,50.80,0.64,0.32,0.16,"(9.9, 24.0)"
2,Andrew Wiggins,espnbet,points,29.5,-165,under,15.46,0.074,0.926,0.623,48.67,0.80,0.40,0.20,"(2.0, 34.4)"
3,Aaron Nesmith,betrivers,points,9.5,-121,under,7.85,0.286,0.714,0.548,30.37,0.37,0.18,0.09,"(2.2, 13.8)"
4,Jonathan Kuminga,espnbet,points,14.5,-140,under,11.03,0.261,0.739,0.583,26.75,0.37,0.19,0.09,"(1.8, 21.3)"
5,Tyrese Maxey,betrivers,points,28.5,-107,under,24.92,0.364,0.636,0.517,23.12,0.25,0.12,0.06,"(7.0, 43.6)"
6,Giannis Antetokounmpo,espnbet,points,24.5,-105,over,27.09,0.622,0.378,0.512,21.40,0.22,0.11,0.06,"(10.9, 43.4)"
7,Andrew Nembhard,betrivers,points,9.5,-122,under,7.64,0.335,0.665,0.550,21.01,0.26,0.13,0.06,"(1.0, 15.9)"
8,Toumani Camara,betrivers,points,10.5,-117,under,8.99,0.349,0.651,0.539,20.74,0.24,0.12,0.06,"(2.2, 16.5)"
9,Tyrese Haliburton,betrivers,points,17.5,-124,under,14.07,0.383,0.617,0.554,11.48,0.14,0.07,0.04,"(1.6, 32.8)"


In [15]:
final_results.to_csv('../DATA/CSV_FILES/BACKTEST_DATA/HISTORICAL/10_27_24.csv')

## Best EVs for 2-leg parlays on prizepicks, underdogs or fanduel

In [ ]:
results = prizepickspairsEV(filterData, odds, model, games, features, espnDate, stake=100, simulations=10000).sort_values(by='EV%', ascending=False).reset_index(drop=True)
print("\nTop 10 highest EV bets:")
results.head()

Processing PrizePicks pairs...

Top 10 highest EV bets:


,PLAYER 1,CATEGORY 1,LINE 1,SIDE 1,PREDICTION 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,LINE 2,SIDE 2,PREDICTION 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,TYPE,PROBABILITY,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER
0,Franz Wagner,points,33.5,under,20.42,0.056,0.944,"(5.4, 36.5)",Clint Capela,points,19.5,under,12.02,0.080,0.920,"(2.5, 22.4)",UNDER/UNDER,0.8680,1.604,0.80,0.40,0.20
1,Franz Wagner,points,33.5,under,20.42,0.056,0.944,"(5.4, 36.5)",Grayson Allen,points,17.5,under,8.66,0.090,0.910,"(0.9, 21.5)",UNDER/UNDER,0.8588,1.576,0.79,0.39,0.20
2,Clint Capela,points,19.5,under,12.02,0.080,0.920,"(2.5, 22.4)",Grayson Allen,points,17.5,under,8.66,0.090,0.910,"(0.9, 21.5)",UNDER/UNDER,0.8368,1.511,0.76,0.38,0.19
3,Franz Wagner,points,33.5,under,20.42,0.056,0.944,"(5.4, 36.5)",Jayson Tatum,points,22.5,over,29.00,0.831,0.169,"(15.6, 42.3)",UNDER/OVER,0.7840,1.352,0.68,0.34,0.17
4,Franz Wagner,points,33.5,under,20.42,0.056,0.944,"(5.4, 36.5)",LaMelo Ball,points,31.5,under,25.16,0.171,0.829,"(12.2, 38.3)",UNDER/UNDER,0.7823,1.347,0.67,0.34,0.17


In [ ]:
def getPlayerSTD(playerDf, prevData, statCol='PTS', std_window=10, minSTD=2.0, maxSTD=9.5):
    if len(s) < 5:
        s = prevData[statCol]
    else:
        s = playerDf[statCol].dropna()
    recentStats = s.tail(std_window) if len(s) >= std_window else s
    std_dev = recentStats.std(ddof=1)

    return float(np.clip(std_dev, minSTD, maxSTD))


def analyzePropBet(player_name, pred, propLine, data, prevData, stat_col='PTS', method='truncated_normal'):
    edge = pred - propLine 
    std_dev = getPlayerSTD(data,prevData,stat_col)

    if method == 'truncated_normal':
        a = -pred / std_dev
        b = np.inf
        probOver = 1 - truncnorm.cdf(propLine, a, b, loc=pred, scale=std_dev)
    else:
        zScore = (propLine - pred) / std_dev 
        probOver = 1 - norm.cdf(zScore)
    
    probUnder = 1 - probOver 

    return {
        'player': player_name,
        'prediction': pred,
        'prop_line': propLine,
        'residual': edge,
        'std_dev': std_dev,
        'prob_over': probOver,
        'prob_under': probUnder,
        'recommendation': 'OVER' if probOver > 0.5 else 'UNDER',
        'confidence': max(probOver, probUnder)
    }
